# Lipkin Results Computation

This notebook runs the selected Lipkin PMM, compression, scaling, and neural-network baseline computations. It saves raw arrays, summary tables, and run settings only. It does not create plots.

This notebook was adapted from a larger original notebook written by myself, with PMM code mostly based on code from Morten. Codex and ChatGPT helped systemize the experiment blocks, expand the run structure, and move repeated helper logic into shared files.


In [7]:
from pathlib import Path
import pickle
import time

import numpy as np
import pandas as pd

from common import (
    GLOBAL_SEED,
    DEFAULT_EPOCHS,
    DEFAULT_LR,
    DEFAULT_INIT_SCALE,
    DEFAULT_Q_GRID,
    DEFAULT_ERR_GRID,
    DEFAULT_R_STEP,
    DEFAULT_REFINE_LOCAL,
    compute_error_stats,
    test_error_summary,
)
from data import make_lipkin_data, make_train_validation_split
from pmm import pmm_parameter_count, compressed_pmm_predictions, train_eigenvalue_pmm_on_data as train_pmm_on_data
from compression import prepare_pmm_compression
from benchmarks import pmm_runtime_row as runtime_row
from baselines import run_mlp_grid_search, train_final_mlp, evaluate_mlp_on_data

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)
LIPKIN_RESULTS_PATH = RESULTS_DIR / "lipkin_results.pkl"

PMM_BASELINE_EPOCHS = 40000
PMM_SWEEP_EPOCHS = 20000
PMM_SCALING_EPOCH_BUDGETS = [5000, 10000, 15000, 20000]
LIPKIN_PMM_SIZES = [64, 96, 128]
LIPKIN_SCALING_PMM_SIZES = [96]
NN_EPOCHS = 2000


def save_results(results, path=LIPKIN_RESULTS_PATH):
    """Save the current result dict to a pickle path and return None."""
    with path.open("wb") as f:
        pickle.dump(results, f, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"Saved Lipkin results to {path}")


def compact_run_for_save(case, compression, data, coordinate_key, regime):
    """Collect representative Lipkin arrays and metadata for saving and plotting.

    Codex and ChatGPT helped separate this compact save format from the
    plotting notebook so results could be reused directly.
    """
    compressed_pred = compressed_pmm_predictions(
        case["M0"],
        case["M1"],
        compression["B_rstar"],
        data["X_test"],
        int(case["k"]),
    )
    return {
        "metadata": {
            "regime": regime,
            "n": int(case["n_pmm"]),
            "k": int(case["k"]),
            "seed": int(case["seed"]),
            "epochs": int(case["epochs"]),
            "r_star": int(compression["r_star"]),
            "r_star_over_k": float(compression["r_star_over_k"]),
            "e_rstar": float(compression["e_rstar"]),
        },
        coordinate_key: np.asarray(data[coordinate_key]),
        "X_test": np.asarray(data["X_test"]),
        "exact": np.asarray(data["E_test"]),
        "pmm": np.asarray(case["pmm_pred"]),
        "compressed_pmm": compressed_pred,
        "pmm_abs_error": np.abs(np.asarray(case["pmm_pred"]) - np.asarray(data["E_test"])),
        "compressed_abs_error": np.abs(compressed_pred - np.asarray(data["E_test"])),
    }


lipkin_results = {}


## 1. Lipkin Baseline Benchmark

Settings: `N = 20`, `n = 10`, `k = 3`, `V in [0, 2]`, using the original baseline PMM training budget. Saves exact/PMM scan arrays, absolute errors, and one compact summary table.


In [8]:
baseline_settings = {
    "N": 20,
    "epsilon": 1.0,
    "V_min": 0.0,
    "V_max": 2.0,
    "n_train": 20,
    "n_test": 100,
    "n_pmm": 10,
    "k": 3,
    "seed": 1,
    "epochs": PMM_BASELINE_EPOCHS,
    "learning_rate": 1e-2,
    "init_scale": 1e-2,
}

baseline_data = make_lipkin_data(
    N=baseline_settings["N"],
    epsilon=baseline_settings["epsilon"],
    Vmin=baseline_settings["V_min"],
    Vmax=baseline_settings["V_max"],
    n_train=baseline_settings["n_train"],
    n_test=baseline_settings["n_test"],
    k_levels=baseline_settings["k"],
)
baseline_case = train_pmm_on_data(
    baseline_data,
    n_pmm=baseline_settings["n_pmm"],
    k=baseline_settings["k"],
    seed=baseline_settings["seed"],
    epochs=baseline_settings["epochs"],
    lr=baseline_settings["learning_rate"],
    init_scale=baseline_settings["init_scale"],
)

baseline_summary = pd.DataFrame([
    {
        "N": baseline_settings["N"],
        "n": baseline_settings["n_pmm"],
        "k": baseline_settings["k"],
        "epochs": baseline_settings["epochs"],
        "V_min": baseline_settings["V_min"],
        "V_max": baseline_settings["V_max"],
        "train_loss": baseline_case["train_loss"],
        "mean_abs_test_error": baseline_case["test_mean_abs_error"],
        "max_abs_test_error": baseline_case["test_max_abs_error"],
        "rms_test_error": baseline_case["test_rms_error"],
        "train_time_s": baseline_case["train_time_s"],
        "best_step": baseline_case["best_step"],
    }
])

lipkin_results["baseline"] = {
    "settings": baseline_settings,
    "summary": baseline_summary,
    "V_test": baseline_data["V_test"],
    "X_test": baseline_data["X_test"],
    "exact": baseline_data["E_test"],
    "pmm": baseline_case["pmm_pred"],
    "abs_error": np.abs(baseline_case["pmm_pred"] - baseline_data["E_test"]),
    "theta_best": baseline_case["theta_best"],
    "M0": baseline_case["M0"],
    "M1": baseline_case["M1"],
}
save_results(lipkin_results)
baseline_summary


Saved Lipkin results to results/lipkin_results.pkl


,N,n,k,epochs,V_min,V_max,train_loss,mean_abs_test_error,max_abs_test_error,rms_test_error,train_time_s,best_step
0,20,10,3,40000,0.0,2.0,0.000001,0.000984,0.002545,0.001129,4.12224,15250


## 2. Lipkin Regime Sweep

Settings: `N = 200`, `k = 3`, three seeds, tolerance `1e-2`, intervals `[0, 2]` and `[0, 4]`, and PMM sizes `n in {64, 96, 128}`.


In [ ]:
regime_sweep_settings = {
    "N": 200,
    "epsilon": 1.0,
    "k": 3,
    "seeds": [GLOBAL_SEED, GLOBAL_SEED + 1, GLOBAL_SEED + 2],
    "tol": 1e-2,
    "n_pmm_values": LIPKIN_PMM_SIZES,
    "intervals": [
        {"name": "V0_2", "V_min": 0.0, "V_max": 2.0, "n_train": 40, "n_test": 200},
        {"name": "V0_4", "V_min": 0.0, "V_max": 4.0, "n_train": 50, "n_test": 200},
    ],
    "epochs": PMM_SWEEP_EPOCHS,
    "learning_rate": DEFAULT_LR,
    "init_scale": DEFAULT_INIT_SCALE,
    "q_grid": DEFAULT_Q_GRID,
    "err_grid": DEFAULT_ERR_GRID,
    "r_step": DEFAULT_R_STEP,
    "refine": DEFAULT_REFINE_LOCAL,
    "representative_selection_metric": "compressed_test_mean_abs_error",
}

sweep_rows = []
runtime_rows = []
run_lookup = {}
representatives = {}

for interval in regime_sweep_settings["intervals"]:
    regime_name = interval["name"]
    data = make_lipkin_data(
        N=regime_sweep_settings["N"],
        epsilon=regime_sweep_settings["epsilon"],
        Vmin=interval["V_min"],
        Vmax=interval["V_max"],
        n_train=interval["n_train"],
        n_test=interval["n_test"],
        k_levels=regime_sweep_settings["k"],
    )

    for n_pmm in regime_sweep_settings["n_pmm_values"]:
        for seed in regime_sweep_settings["seeds"]:
            print(f"Lipkin sweep: regime={regime_name}, n={n_pmm}, seed={seed}")
            case = train_pmm_on_data(
                data,
                n_pmm=n_pmm,
                k=regime_sweep_settings["k"],
                seed=seed,
                epochs=regime_sweep_settings["epochs"],
                lr=regime_sweep_settings["learning_rate"],
                init_scale=regime_sweep_settings["init_scale"],
            )
            comp = prepare_pmm_compression(
                case,
                tol=regime_sweep_settings["tol"],
                n_grid_Q=regime_sweep_settings["q_grid"],
                n_grid_err=regime_sweep_settings["err_grid"],
                r_step=regime_sweep_settings["r_step"],
                refine=regime_sweep_settings["refine"],
            )
            compressed_pred = compressed_pmm_predictions(case["M0"], case["M1"], comp["B_rstar"], data["X_test"], regime_sweep_settings["k"])
            compressed_stats = test_error_summary(data["E_test"], compressed_pred)

            row = {
                "regime": regime_name,
                "V_min": interval["V_min"],
                "V_max": interval["V_max"],
                "N": regime_sweep_settings["N"],
                "n": int(n_pmm),
                "k": regime_sweep_settings["k"],
                "seed": int(seed),
                "epochs": int(case["epochs"]),
                "train_loss": case["train_loss"],
                "test_loss": case["test_loss"],
                "test_mean_abs_error": case["test_mean_abs_error"],
                "test_max_abs_error": case["test_max_abs_error"],
                "test_rms_error": case["test_rms_error"],
                "compressed_test_loss": compressed_stats["test_loss"],
                "compressed_test_mean_abs_error": compressed_stats["test_mean_abs_error"],
                "compressed_test_max_abs_error": compressed_stats["test_max_abs_error"],
                "compressed_test_rms_error": compressed_stats["test_rms_error"],
                "train_time_s": case["train_time_s"],
                "r_star": comp["r_star"],
                "r_star_over_k": comp["r_star_over_k"],
                "e_rstar": comp["e_rstar"],
                "e_2k": comp["e_2k"],
                "compression_preprocess_time_s": comp["compression_preprocess_time_s"],
            }
            sweep_rows.append(row)
            runtime_rows.append(runtime_row(case, comp, regime=regime_name, n_deploy=len(data["X_test"])))

            key = (regime_name, int(n_pmm), int(seed))
            run_lookup[key] = compact_run_for_save(case, comp, data, coordinate_key="V_test", regime=regime_name)

sweep_detail = pd.DataFrame(sweep_rows)
regime_summary = (
    sweep_detail.groupby(["regime", "V_min", "V_max", "n", "epochs"], as_index=False)
    .agg(
        mean_test_loss=("test_loss", "mean"),
        std_test_loss=("test_loss", "std"),
        mean_train_time_s=("train_time_s", "mean"),
        mean_r_star=("r_star", "mean"),
        mean_r_star_over_k=("r_star_over_k", "mean"),
        mean_e_rstar=("e_rstar", "mean"),
        mean_test_mean_abs_error=("test_mean_abs_error", "mean"),
        mean_compressed_test_mean_abs_error=("compressed_test_mean_abs_error", "mean"),
    )
    .sort_values(["regime", "mean_test_loss", "n"])
    .reset_index(drop=True)
)
regime_summary["std_test_loss"] = regime_summary["std_test_loss"].fillna(0.0)
runtime_detail = pd.DataFrame(runtime_rows)
runtime_summary = (
    runtime_detail.groupby(["regime", "n", "epochs"], as_index=False)
    .agg(
        mean_r_star=("r_star", "mean"),
        mean_r_star_over_k=("r_star_over_k", "mean"),
        mean_e_rstar=("e_rstar", "mean"),
        pmm_scan_time_s=("pmm_scan_time_s", "mean"),
        pmm_compressed_scan_time_s=("pmm_compressed_scan_time_s", "mean"),
        compression_preprocess_time_s=("compression_preprocess_time_s", "mean"),
        total_reduced_time_s=("total_reduced_time_s", "mean"),
        break_even_scan_count=("break_even_scan_count", "mean"),
        raw_speedup=("raw_speedup", "mean"),
        amortized_speedup=("amortized_speedup", "mean"),
    )
    .sort_values(["regime", "n"])
    .reset_index(drop=True)
)

representative_metric = regime_sweep_settings["representative_selection_metric"]
for regime_name in sweep_detail["regime"].unique():
    best = sweep_detail[sweep_detail["regime"].eq(regime_name)].sort_values(representative_metric).iloc[0]
    representative = run_lookup[(best["regime"], int(best["n"]), int(best["seed"]))]
    representative["metadata"]["selection_metric"] = representative_metric
    representatives[regime_name] = representative

lipkin_results["regime_sweep"] = {
    "settings": regime_sweep_settings,
    "detail": sweep_detail,
    "summary": regime_summary,
    "runtime_detail": runtime_detail,
    "runtime_summary": runtime_summary,
    "representatives": representatives,
    "runs_by_key": run_lookup,
}
save_results(lipkin_results)
regime_summary


In [9]:
completed_regimes = ["V0_2"]

partial_sweep_rows = [
    row for row in sweep_rows
    if row["regime"] in completed_regimes
]
partial_runtime_rows = [
    row for row in runtime_rows
    if row["regime"] in completed_regimes
]
partial_run_lookup = {
    key: value for key, value in run_lookup.items()
    if key[0] in completed_regimes
}

sweep_detail = pd.DataFrame(partial_sweep_rows)
runtime_detail = pd.DataFrame(partial_runtime_rows)

regime_summary = (
    sweep_detail.groupby(["regime", "V_min", "V_max", "n", "epochs"], as_index=False)
    .agg(
        mean_test_loss=("test_loss", "mean"),
        std_test_loss=("test_loss", "std"),
        mean_train_time_s=("train_time_s", "mean"),
        mean_r_star=("r_star", "mean"),
        mean_r_star_over_k=("r_star_over_k", "mean"),
        mean_e_rstar=("e_rstar", "mean"),
        mean_test_mean_abs_error=("test_mean_abs_error", "mean"),
        mean_compressed_test_mean_abs_error=("compressed_test_mean_abs_error", "mean"),
    )
    .sort_values(["regime", "mean_test_loss", "n"])
    .reset_index(drop=True)
)
regime_summary["std_test_loss"] = regime_summary["std_test_loss"].fillna(0.0)

runtime_summary = (
    runtime_detail.groupby(["regime", "n", "epochs"], as_index=False)
    .agg(
        mean_r_star=("r_star", "mean"),
        mean_r_star_over_k=("r_star_over_k", "mean"),
        mean_e_rstar=("e_rstar", "mean"),
        pmm_scan_time_s=("pmm_scan_time_s", "mean"),
        pmm_compressed_scan_time_s=("pmm_compressed_scan_time_s", "mean"),
        compression_preprocess_time_s=("compression_preprocess_time_s", "mean"),
        total_reduced_time_s=("total_reduced_time_s", "mean"),
        break_even_scan_count=("break_even_scan_count", "mean"),
        raw_speedup=("raw_speedup", "mean"),
        amortized_speedup=("amortized_speedup", "mean"),
    )
    .sort_values(["regime", "n"])
    .reset_index(drop=True)
)

representatives = {}
representative_metric = regime_sweep_settings["representative_selection_metric"]
for regime_name in completed_regimes:
    best = sweep_detail[sweep_detail["regime"].eq(regime_name)].sort_values(representative_metric).iloc[0]
    representative = partial_run_lookup[(best["regime"], int(best["n"]), int(best["seed"]))]
    representative["metadata"]["selection_metric"] = representative_metric
    representatives[regime_name] = representative

lipkin_results["regime_sweep"] = {
    "settings": {**regime_sweep_settings, "intervals": [regime_sweep_settings["intervals"][0]]},
    "detail": sweep_detail,
    "summary": regime_summary,
    "runtime_detail": runtime_detail,
    "runtime_summary": runtime_summary,
    "representatives": representatives,
    "runs_by_key": partial_run_lookup,
}

save_results(lipkin_results)
regime_summary


Saved Lipkin results to results/lipkin_results.pkl


,regime,V_min,V_max,n,epochs,mean_test_loss,std_test_loss,mean_train_time_s,mean_r_star,mean_r_star_over_k,mean_e_rstar,mean_test_mean_abs_error,mean_compressed_test_mean_abs_error
0,V0_2,0.0,2.0,128,20000,0.000375,0.000058,1543.061681,10.000000,3.333333,0.000369,0.014457,0.014466
1,V0_2,0.0,2.0,96,20000,0.000447,0.000118,956.750994,9.333333,3.111111,0.003428,0.015737,0.015791
2,V0_2,0.0,2.0,64,20000,0.001520,0.000403,206.195232,8.000000,2.666667,0.006081,0.030579,0.030552


## 3. Lipkin PMM Size / Epoch Scaling

To save on runetime, we run n = 96 with 5k, 10k, 15k and 20k epochs. Hardcoded so that it does NOT choose the best run from the previous training ( the  n = 128 PMM), byt unstead chooses the n = 96 PMM which gave very comparable results. 


In [15]:
best_regime_row = regime_summary.sort_values("mean_test_loss").iloc[0]
best_interval = next(item for item in regime_sweep_settings["intervals"] if item["name"] == best_regime_row["regime"])

scaling_settings = {
    "source": "best_lipkin_regime_from_regime_sweep",
    "best_regime": best_regime_row["regime"],
    "N": regime_sweep_settings["N"],
    "epsilon": regime_sweep_settings["epsilon"],
    "V_min": float(best_interval["V_min"]),
    "V_max": float(best_interval["V_max"]),
    "n_train": int(best_interval["n_train"]),
    "n_test": int(best_interval["n_test"]),
    "k": regime_sweep_settings["k"],
    "seed": GLOBAL_SEED,
    "diagnostic_type": "single_seed_scaling",
    # Hard-coded so this diagnostic does not inherit the best sweep size.
    "n_pmm_values": [96],
    "epoch_budgets": [5000, 10000, 15000, 20000],
    "learning_rate": DEFAULT_LR,
    "init_scale": DEFAULT_INIT_SCALE,
}

scaling_data = make_lipkin_data(
    N=scaling_settings["N"],
    epsilon=scaling_settings["epsilon"],
    Vmin=scaling_settings["V_min"],
    Vmax=scaling_settings["V_max"],
    n_train=scaling_settings["n_train"],
    n_test=scaling_settings["n_test"],
    k_levels=scaling_settings["k"],
)

scaling_rows = []
scaling_predictions = {}
print("Lipkin scaling settings:", scaling_settings)
for n_pmm in scaling_settings["n_pmm_values"]:
    for epochs in scaling_settings["epoch_budgets"]:
        print(f"Lipkin scaling: n={n_pmm}, epochs={epochs}")
        case = train_pmm_on_data(
            scaling_data,
            n_pmm=n_pmm,
            k=scaling_settings["k"],
            seed=scaling_settings["seed"],
            epochs=epochs,
            lr=scaling_settings["learning_rate"],
            init_scale=scaling_settings["init_scale"],
        )
        scaling_rows.append({
            "n": int(n_pmm),
            "epochs": int(epochs),
            "test_mean_abs_error": case["test_mean_abs_error"],
            "test_max_abs_error": case["test_max_abs_error"],
            "test_rms_error": case["test_rms_error"],
            "test_loss": case["test_loss"],
            "train_loss": case["train_loss"],
            "train_time_s": case["train_time_s"],
            "best_step": case["best_step"],
        })
        scaling_predictions[(int(n_pmm), int(epochs))] = {
            "V_test": scaling_data["V_test"],
            "exact": scaling_data["E_test"],
            "pmm": case["pmm_pred"],
            "abs_error": np.abs(case["pmm_pred"] - scaling_data["E_test"]),
        }

scaling_detail = pd.DataFrame(scaling_rows).sort_values(["n", "epochs"]).reset_index(drop=True)
scaling_error_table = scaling_detail[["n", "epochs", "test_mean_abs_error", "test_max_abs_error", "test_rms_error", "test_loss"]]
scaling_train_time_table = scaling_detail[["n", "epochs", "train_time_s", "train_loss", "best_step"]]

lipkin_results["pmm_size_epoch_scaling"] = {
    "settings": scaling_settings,
    "detail": scaling_detail,
    "error_table": scaling_error_table,
    "train_time_table": scaling_train_time_table,
    "predictions_by_key": scaling_predictions,
}
save_results(lipkin_results)
scaling_detail


Lipkin scaling settings: {'source': 'best_lipkin_regime_from_regime_sweep', 'best_regime': 'V0_2', 'N': 200, 'epsilon': 1.0, 'V_min': 0.0, 'V_max': 2.0, 'n_train': 40, 'n_test': 200, 'k': 3, 'seed': 1234, 'diagnostic_type': 'single_seed_scaling', 'n_pmm_values': [96], 'epoch_budgets': [5000, 10000, 15000, 20000], 'learning_rate': 0.01, 'init_scale': 0.01}
Lipkin scaling: n=96, epochs=5000
Lipkin scaling: n=96, epochs=10000
Lipkin scaling: n=96, epochs=15000
Lipkin scaling: n=96, epochs=20000
Saved Lipkin results to results/lipkin_results.pkl


,n,epochs,test_mean_abs_error,test_max_abs_error,test_rms_error,test_loss,train_loss,train_time_s,best_step
0,96,5000,0.486340,1.332550,0.568711,0.323432,0.334522,237.259426,5000
1,96,10000,0.367527,0.830391,0.420547,0.176860,0.185335,433.296764,10000
2,96,15000,0.059333,0.187014,0.071443,0.005104,0.005355,1972.222961,15000
3,96,20000,0.013542,0.051965,0.017707,0.000314,0.000331,1972.896469,20000


## 4. Lipkin Raw PMM vs Neural Network

Harcoded to use n = 96 PMM to save on runtime.


In [17]:
pmm_vs_nn_selection_metric = "compressed_test_mean_abs_error"
pmm_vs_nn_pmm_size = 96
pmm_vs_nn_candidates = sweep_detail[sweep_detail["n"].eq(pmm_vs_nn_pmm_size)].copy()
best_run_row = pmm_vs_nn_candidates.sort_values(pmm_vs_nn_selection_metric).iloc[0]
best_run_key = (best_run_row["regime"], int(best_run_row["n"]), int(best_run_row["seed"]))
best_pmm_run = run_lookup[best_run_key]
best_interval = next(item for item in regime_sweep_settings["intervals"] if item["name"] == best_run_row["regime"])

nn_settings = {
    "source": "hardcoded_n96_compression_selected_raw_pmm_vs_nn",
    "regime": best_run_row["regime"],
    "N": regime_sweep_settings["N"],
    "epsilon": regime_sweep_settings["epsilon"],
    "V_min": float(best_interval["V_min"]),
    "V_max": float(best_interval["V_max"]),
    "n_train": int(best_interval["n_train"]),
    "n_test": int(best_interval["n_test"]),
    "k": regime_sweep_settings["k"],
    "pmm_n": int(best_run_row["n"]),
    "pmm_hardcoded_n": int(pmm_vs_nn_pmm_size),
    "pmm_seed": int(best_run_row["seed"]),
    "pmm_epochs": int(best_run_row["epochs"]),
    "pmm_selection_metric": pmm_vs_nn_selection_metric,
    "nn_widths": [32, 64, 128],
    "nn_activations": ["tanh", "relu"],
    "nn_weight_decays": [0.0, 1e-5],
    "nn_lr": 1e-3,
    "nn_epochs": NN_EPOCHS,
    "nn_batch_size": 16,
    "nn_seed": GLOBAL_SEED,
    "device": "cpu",
}

nn_data = make_lipkin_data(
    N=nn_settings["N"],
    epsilon=nn_settings["epsilon"],
    Vmin=nn_settings["V_min"],
    Vmax=nn_settings["V_max"],
    n_train=nn_settings["n_train"],
    n_test=nn_settings["n_test"],
    k_levels=nn_settings["k"],
)
(
    X_train,
    y_train,
    X_val,
    y_val,
    train_idx,
    val_idx,
) = make_train_validation_split(nn_data["X_train"], nn_data["y_train"], val_fraction=0.25)

nn_grid_summary, best_nn = run_mlp_grid_search(
    X_train,
    y_train,
    X_val,
    y_val,
    widths=tuple(nn_settings["nn_widths"]),
    activations=tuple(nn_settings["nn_activations"]),
    weight_decays=tuple(nn_settings["nn_weight_decays"]),
    lr=nn_settings["nn_lr"],
    epochs=nn_settings["nn_epochs"],
    batch_size=nn_settings["nn_batch_size"],
    seed=nn_settings["nn_seed"],
    device=nn_settings["device"],
)

nn_final = train_final_mlp(
    nn_data["X_train"],
    nn_data["y_train"],
    hidden_width=best_nn["hidden_width"],
    activation=best_nn["activation"],
    weight_decay=best_nn["weight_decay"],
    lr=nn_settings["nn_lr"],
    epochs=nn_settings["nn_epochs"],
    batch_size=nn_settings["nn_batch_size"],
    seed=nn_settings["nn_seed"],
    device=nn_settings["device"],
)
nn_eval = evaluate_mlp_on_data(nn_final["model"], nn_data["X_test"], nn_data["y_test"], device=nn_settings["device"])

pmm_pred = best_pmm_run["pmm"]
nn_pred = nn_eval["y_pred"]

pmm_final_train_time_s = float(best_run_row["train_time_s"])
pmm_model_selection_time_best_regime_s = float(
    sweep_detail.loc[sweep_detail["regime"].eq(best_run_row["regime"]), "train_time_s"].sum()
)
pmm_model_selection_time_all_sweep_s = float(sweep_detail["train_time_s"].sum())
pmm_total_prep_time_s = pmm_model_selection_time_best_regime_s
pmm_test_loss = float(best_run_row["test_loss"])
pmm_train_loss = float(best_run_row["train_loss"])
pmm_parameter_count_value = pmm_parameter_count(nn_settings["pmm_n"], n_features=1)

nn_model_selection_time_s = float(nn_grid_summary["train_time"].sum())
nn_final_train_time_s = float(nn_final["train_time"])
nn_total_prep_time_s = nn_model_selection_time_s + nn_final_train_time_s

comparison_rows = []
for model_label, pred, parameter_count, epochs_used, final_train_time, model_selection_time, total_prep_time in [
    (
        "PMM",
        pmm_pred,
        pmm_parameter_count_value,
        int(best_run_row["epochs"]),
        pmm_final_train_time_s,
        pmm_model_selection_time_best_regime_s,
        pmm_total_prep_time_s,
    ),
    (
        "NN",
        nn_pred,
        int(nn_final["parameter_count"]),
        int(nn_settings["nn_epochs"]),
        nn_final_train_time_s,
        nn_model_selection_time_s,
        nn_total_prep_time_s,
    ),
]:
    stats = compute_error_stats(nn_data["E_test"], pred)
    comparison_rows.append({
        "model": model_label,
        "mean_abs_test_error": stats["mean_abs_error"],
        "max_abs_test_error": stats["max_abs_error"],
        "rms_test_error": stats["rms_error"],
        "parameter_count": int(parameter_count),
        "epochs": int(epochs_used),
        "final_train_time_s": float(final_train_time),
        "model_selection_time_s": float(model_selection_time),
        "total_prep_time_s": float(total_prep_time),
    })
comparison_table = pd.DataFrame(comparison_rows)

lipkin_results["pmm_vs_nn"] = {
    "settings": nn_settings,
    "comparison_table": comparison_table,
    "pmm_metadata": {
        "regime": best_run_row["regime"],
        "n": int(best_run_row["n"]),
        "seed": int(best_run_row["seed"]),
        "epochs": int(best_run_row["epochs"]),
        "k": int(best_run_row["k"]),
        "selection_metric": pmm_vs_nn_selection_metric,
        "hardcoded_n": int(pmm_vs_nn_pmm_size),
        "parameter_count": int(pmm_parameter_count_value),
        "train_time_s": pmm_final_train_time_s,
        "train_loss": pmm_train_loss,
        "raw_test_loss": pmm_test_loss,
        "raw_test_mean_abs_error": float(best_run_row["test_mean_abs_error"]),
        "raw_test_max_abs_error": float(best_run_row["test_max_abs_error"]),
        "raw_test_rms_error": float(best_run_row["test_rms_error"]),
        "compressed_test_loss": float(best_run_row["compressed_test_loss"]),
        "compressed_test_mean_abs_error": float(best_run_row["compressed_test_mean_abs_error"]),
        "compressed_test_max_abs_error": float(best_run_row["compressed_test_max_abs_error"]),
        "compressed_test_rms_error": float(best_run_row["compressed_test_rms_error"]),
        "model_selection_time_best_regime_s": pmm_model_selection_time_best_regime_s,
        "model_selection_time_all_sweep_s": pmm_model_selection_time_all_sweep_s,
        "total_prep_time_s": pmm_total_prep_time_s,
    },
    "timing_notes": {
        "final_train_time_s": "Time for the final selected model training run.",
        "pmm_selection_metric": "The representative PMM run is selected only among n=96 sweep runs by compressed PMM quality; the main comparison table compares raw PMM predictions against NN predictions.",
        "model_selection_time_s": "For PMM, summed PMM sweep train time within the selected regime. For NN, summed grid-search train time.",
        "total_prep_time_s": "Model-selection time plus any separate final retraining time. The PMM selected run is reused from the sweep, so there is no extra final retrain.",
    },
    "nn_grid_summary": nn_grid_summary,
    "nn_best_validation": {k: v for k, v in best_nn.items() if k not in {"model", "y_val_pred"}},
    "nn_final_summary": {k: v for k, v in nn_final.items() if k != "model"},
    "V_test": nn_data["V_test"],
    "exact": nn_data["E_test"],
    "pmm": pmm_pred,
    "compressed_pmm": best_pmm_run["compressed_pmm"],
    "nn": nn_pred,
    "pmm_abs_error": np.abs(pmm_pred - nn_data["E_test"]),
    "compressed_pmm_abs_error": np.abs(best_pmm_run["compressed_pmm"] - nn_data["E_test"]),
    "nn_abs_error": np.abs(nn_pred - nn_data["E_test"]),
    "train_validation_indices": {"train_idx": train_idx, "val_idx": val_idx},
}
save_results(lipkin_results)
comparison_table


Saved Lipkin results to results/lipkin_results.pkl


,model,mean_abs_test_error,max_abs_test_error,rms_test_error,parameter_count,epochs,final_train_time_s,model_selection_time_s,total_prep_time_s
0,PMM,0.013542,0.051965,0.017707,18432,20000,1072.511549,8118.023723,8118.023723
1,NN,0.042723,0.230021,0.060692,17155,2000,2.049159,16.351482,18.400641
